### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** The two-tower model is the DOMINANT architecture in production RecSys today. It's also the same architecture used in semantic search, RAG retrieval (NB28), and CLIP (multimodal). Understanding two-tower gives you one mental model for recommendations, search, and retrieval.

**Mental Model:** Imagine two experts: one who deeply understands the USER (their history, demographics, context), and another who deeply understands each ITEM (content, metadata, popularity). Each expert writes a summary (embedding vector). A recommendation is good if both summaries "agree" (high dot product). This is the two-tower model.

**Common Junior Pitfall:** Training a neural RecSys on only positive interactions (clicks, purchases). Without NEGATIVE examples, the model learns that everything is relevant. Production systems generate hard negatives: items the user SAW but didn't click.

---

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
* [From Dot Products to Neural Networks](#from-dot-products-to-neural-networks)
  * [🎓 Junior to Senior: Concept Bridge](#-junior-to-senior-concept-bridge)
* [1 · Neural Collaborative Filtering (NCF)](#1-neural-collaborative-filtering-ncf)
* [2 · Two-Tower Model](#2-two-tower-model)
  * [The Production Standard](#the-production-standard)
  * [Why Two Towers?](#why-two-towers)
* [3 · Production RecSys Architecture](#3-production-recsys-architecture)
  * [The Funnel](#the-funnel)
  * [🎓 DEEP QUESTION ANSWERED](#-deep-question-answered)
* [✅ Knowledge Check](#-knowledge-check)
  * [Q1: Two-tower vs NCF](#q1-two-tower-vs-ncf)
  * [Q2: Negative sampling](#q2-negative-sampling)
  * [Q3: Feature engineering](#q3-feature-engineering)
* [🔨 Practical Practice](#-practical-practice)
  * [Exercise 1: In-Batch Negatives](#exercise-1-in-batch-negatives)
  * [Exercise 2: Wide & Deep](#exercise-2-wide-deep)
  * [Exercise 3: Attention over History](#exercise-3-attention-over-history)

---


In [ ]:
!pip install -q torch numpy matplotlib

## 1 · Neural Collaborative Filtering (NCF)

Replace the dot product with a neural network that can learn NON-LINEAR interactions:

```
User ID  →  [Embedding]  →  u (64-dim)
                                ↘
                                 [Concat]  →  [MLP layers]  →  predicted rating
                                ↗
Item ID  →  [Embedding]  →  v (64-dim)
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

class NeuralCF(nn.Module):
    """Neural Collaborative Filtering (He et al., 2017).
    
    Combines GMF (Generalized MF) + MLP for nonlinear interactions.
    GMF path: element-wise product of embeddings (like MF)
    MLP path: concatenated embeddings through hidden layers (nonlinear)
    """
    
    def __init__(self, n_users, n_items, embed_dim=32, hidden_dims=[64, 32, 16]):
        super().__init__()
        
        # GMF path embeddings
        self.user_embed_gmf = nn.Embedding(n_users, embed_dim)
        self.item_embed_gmf = nn.Embedding(n_items, embed_dim)
        
        # MLP path embeddings
        self.user_embed_mlp = nn.Embedding(n_users, embed_dim)
        self.item_embed_mlp = nn.Embedding(n_items, embed_dim)
        
        # MLP layers
        mlp_layers = []
        input_dim = embed_dim * 2  # Concatenated user + item
        for hidden_dim in hidden_dims:
            mlp_layers.extend([
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
            ])
            input_dim = hidden_dim
        self.mlp = nn.Sequential(*mlp_layers)
        
        # Output: combine GMF and MLP paths
        self.output = nn.Linear(embed_dim + hidden_dims[-1], 1)
    
    def forward(self, user_ids, item_ids):
        # GMF path: element-wise product
        gmf_out = self.user_embed_gmf(user_ids) * self.item_embed_gmf(item_ids)
        
        # MLP path: concatenate then transform
        user_mlp = self.user_embed_mlp(user_ids)
        item_mlp = self.item_embed_mlp(item_ids)
        mlp_input = torch.cat([user_mlp, item_mlp], dim=-1)
        mlp_out = self.mlp(mlp_input)
        
        # Combine both paths
        combined = torch.cat([gmf_out, mlp_out], dim=-1)
        return self.output(combined).squeeze(-1)

# Create training data from sparse matrix
np.random.seed(42)
n_users, n_items = 200, 50

# Generate data with patterns
true_U = np.random.randn(n_users, 5) * 0.5
true_V = np.random.randn(n_items, 5) * 0.5
true_R = 3.0 + true_U @ true_V.T + np.random.randn(n_users, n_items) * 0.3
true_R = np.clip(true_R, 1, 5)
mask = np.random.random((n_users, n_items)) < 0.10

# Training pairs
train_users, train_items = np.where(mask)
train_ratings = true_R[mask].astype(np.float32)

# Train NCF
model = NeuralCF(n_users, n_items, embed_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(100):
    model.train()
    users_t = torch.LongTensor(train_users)
    items_t = torch.LongTensor(train_items)
    ratings_t = torch.FloatTensor(train_ratings)
    
    preds = model(users_t, items_t)
    loss = loss_fn(preds, ratings_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f'NCF Architecture: {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'Training RMSE: {np.sqrt(losses[-1]):.3f}')

plt.figure(figsize=(8, 4))
plt.plot(np.sqrt(losses), lw=2, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('Neural Collaborative Filtering Training')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2 · Two-Tower Model

### The Production Standard

The two-tower model separates user and item encoding into INDEPENDENT towers:

```
User Tower                              Item Tower
┌──────────────────┐                   ┌──────────────────┐
│ User ID          │                   │ Item ID          │
│ Age, Gender      │                   │ Genre, Year      │
│ Watch History    │                   │ Description      │
│ Device, Time     │                   │ Thumbnail embed  │
└────────┬─────────┘                   └────────┬─────────┘
         │                                      │
    [MLP layers]                           [MLP layers]
         │                                      │
    user_embed (128-d)                    item_embed (128-d)
         │                                      │
         └──────────── dot product ─────────────┘
                           │
                     relevance score
```

### Why Two Towers?

| Advantage | Why |
|-----------|-----|
| **Pre-compute item embeddings** | Compute once, serve millions of times |
| **ANN retrieval** | Find top-K from millions in <10ms (HNSW, NB27) |
| **Feature richness** | Each tower can use different feature types |
| **Scale** | YouTube serves 1B+ users with this architecture |

In [ ]:
import torch
import torch.nn as nn

class TwoTowerModel(nn.Module):
    """Two-Tower Retrieval Model.
    
    Used by: YouTube, TikTok, Spotify, Airbnb.
    Same architecture as: CLIP (image+text), DPR (query+passage).
    
    Key insight: item embeddings are pre-computed offline.
    At serving time, only the user tower runs + ANN search.
    This makes retrieval from millions of items sub-10ms.
    """
    
    def __init__(self, n_users, n_items, embed_dim=64, output_dim=32):
        super().__init__()
        
        # User tower
        self.user_tower = nn.Sequential(
            nn.Embedding(n_users, embed_dim),
        )
        self.user_mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim),
            nn.functional.normalize,  # L2 normalize
        )
        
        # Item tower  
        self.item_tower = nn.Sequential(
            nn.Embedding(n_items, embed_dim),
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim),
        )
    
    def encode_user(self, user_ids):
        x = self.user_tower[0](user_ids)
        x = nn.functional.relu(nn.functional.linear(x, self.user_mlp[0].weight, self.user_mlp[0].bias))
        x = nn.functional.linear(x, self.user_mlp[2].weight, self.user_mlp[2].bias)
        return nn.functional.normalize(x, dim=-1)
    
    def encode_item(self, item_ids):
        x = self.item_tower[0](item_ids)
        x = nn.functional.relu(nn.functional.linear(x, self.item_mlp[0].weight, self.item_mlp[0].bias))
        x = nn.functional.linear(x, self.item_mlp[2].weight, self.item_mlp[2].bias)
        return nn.functional.normalize(x, dim=-1)
    
    def forward(self, user_ids, item_ids):
        user_emb = self.encode_user(user_ids)
        item_emb = self.encode_item(item_ids)
        # Dot product = relevance score
        return (user_emb * item_emb).sum(dim=-1)

# Architecture summary
model_tt = TwoTowerModel(n_users=1_000_000, n_items=100_000, embed_dim=64, output_dim=32)
total_params = sum(p.numel() for p in model_tt.parameters())

print(f'Two-Tower Architecture:')
print(f'  Total parameters: {total_params:,}')
print(f'  User tower: user_id → 64-dim embed → MLP → 32-dim normalized')
print(f'  Item tower: item_id → 64-dim embed → MLP → 32-dim normalized')
print(f'  Score: dot_product(user_embed, item_embed)')
print(f'\nServing pipeline:')
print(f'  1. OFFLINE: pre-compute all 100K item embeddings')
print(f'  2. OFFLINE: index item embeddings in HNSW (NB27)')
print(f'  3. ONLINE: compute user embedding (1 forward pass, <1ms)')
print(f'  4. ONLINE: ANN search for top-100 nearest items (<5ms)')
print(f'  5. ONLINE: re-rank top-100 with a richer model (<10ms)')

---
## 3 · Production RecSys Architecture

### The Funnel

No single model handles the full pipeline. Production systems use a FUNNEL:

```
All Items (100M+)
       │
  [Candidate Generation]    Two-Tower + ANN → top 1000    ~10ms
       │
  [Ranking/Scoring]         Feature-rich model → top 100   ~20ms
       │
  [Re-Ranking]              Business rules, diversity      ~5ms
       │
  [Shown to User]           Final 10-20 items
```

| Stage | Model | Speed | Quality | Input |
|-------|-------|-------|---------|-------|
| Candidate Gen | Two-Tower | <10ms | Medium | User embedding only |
| Ranking | Wide & Deep / DLRM | <20ms | High | Rich features (100+ features) |
| Re-Ranking | Rule-based | <5ms | - | Diversity, freshness, business rules |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How do neural networks capture nonlinear interactions?*

**A:** MF computes $u \cdot v$ — this can only capture linear correlations between latent factors. NCF replaces this with an MLP that takes the concatenated [u, v] vector and applies nonlinear activations (ReLU). This lets the model learn patterns like "users who like BOTH action AND romance prefer romantic action movies, but users who like only action prefer pure action movies." The MLP can express any function of the combined user-item representation.

---

## ✅ Knowledge Check

### Q1: Two-tower vs NCF
Why can't NCF be used for candidate generation from 100M items?

<details><summary>👉 Answer</summary>

NCF concatenates user and item embeddings before passing through the MLP. This means you can't pre-compute item representations — the MLP must run for EVERY (user, item) pair. For 100M items, that's 100M forward passes per user request. Two-tower separates the computation: item embeddings are pre-computed once, enabling fast ANN retrieval.
</details>

### Q2: Negative sampling
In implicit feedback (clicks/views), we only have POSITIVE interactions. Why do we need negative examples?

<details><summary>👉 Answer</summary>

Without negatives, the model thinks EVERYTHING is relevant (it only sees positive examples). We need negatives to teach "these items are NOT relevant." Hard negatives (items the user saw but didn't click) are more informative than random negatives (items the user never saw). Production systems mix: 50% random negatives (easy) + 50% in-batch negatives (hard — other users' positives).
</details>

### Q3: Feature engineering
YouTube's ranking model uses 100+ features. Name 5 categories.

<details><summary>👉 Answer</summary>

(1) User demographics (age, country, device), (2) Viewing history (last 50 videos, watch time distribution), (3) Video features (duration, upload time, channel, tags), (4) Context (time of day, day of week, device type), (5) Cross-features (has user watched this channel before? similar video watch count). Real-time features from Feature Store (NB08) are critical.
</details>

---

## 🔨 Practical Practice

### Exercise 1: In-Batch Negatives
Implement in-batch negative sampling: in each training batch, use other users' positive items as negatives. This is the most efficient negative sampling strategy.

### Exercise 2: Wide & Deep
Implement Google's Wide & Deep model: a linear model ("wide" - memorizes feature crosses) combined with an MLP ("deep" - generalizes).

### Exercise 3: Attention over History
Instead of a static user embedding, implement attention over the user's last 20 item interactions. The query is the candidate item — items in history relevant to the candidate get higher attention weight.

---

**Next →** RS 04: Retrieval, Ranking & Real-Time Systems